# Module 5: LangChain
# Topics 48 & 49: Structured Output & Streaming

> **Interview Difficulty:** ⭐⭐⭐⭐⭐
>
> **Interview Frequency:** Extremely High
>
> **Production Importance:** Critical
>
> **Prerequisites:**
> - Output Parsers ✅
> - LCEL ✅
> - Agents ✅

---

# Learning Objectives

After this topic, you should be able to answer:

- What is Structured Output?
- Why not parse plain text?
- JSON Mode vs Structured Output
- Pydantic Models
- Output Parsers vs Structured Output
- What is Streaming?
- How Streaming works
- Streaming Events
- Best Practices
- Interview Questions

---

# Part 1 : Structured Output

# 1. What is Structured Output?

Normally an LLM returns plain text.

Example

```
John is 28 years old.
```

Suppose our application needs

```json
{
  "name":"John",
  "age":28
}
```

This is where Structured Output comes in.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **Structured Output is the process of constraining an LLM to return data in a predefined schema such as JSON or a typed object, enabling reliable downstream processing.**

---

# 2. Why Do We Need Structured Output?

Suppose we build

- Resume Parser
- Invoice Extractor
- Medical Report Analyzer
- Order Processing System

The application cannot reliably process

```
John is 28 years old.
```

Instead it needs

```json
{
 "name":"John",
 "age":28
}
```

---

# 3. Without Structured Output

```text
User

↓

LLM

↓

Free Text

↓

Regex

↓

Sometimes Works

↓

Sometimes Breaks
```

---

# 4. With Structured Output

```text
User

↓

LLM

↓

Validated JSON

↓

Application

↓

Database/API
```

Much more reliable.

---

# 5. Pydantic Schema

```python
from pydantic import BaseModel

class Employee(BaseModel):
    name: str
    age: int
    department: str
```

This defines the expected response structure.

---

# 6. LangChain Structured Output

Example

```python
structured_llm = llm.with_structured_output(
    Employee
)

response = structured_llm.invoke(
    "John is 28 years old and works in HR."
)

print(response)
```

Output

```python
Employee(
    name="John",
    age=28,
    department="HR"
)
```

---

# 7. Internal Workflow

```text
Prompt

↓

LLM

↓

Schema Validation

↓

Structured Object

↓

Application
```

---

# 8. Structured Output vs Output Parser

Very common interview question.

| Output Parser | Structured Output |
|---------------|-------------------|
| Parses LLM output | Model generates schema directly |
| Can fail on invalid text | More reliable |
| Often prompt-based | Uses schema enforcement (when supported) |
| Extra parsing step | Native structured response |

---

# 9. Structured Output vs JSON Mode

| JSON Mode | Structured Output |
|------------|------------------|
| Valid JSON | Valid JSON + Schema |
| No type validation | Strong typing |
| Missing fields possible | Expected fields validated |
| Less strict | More reliable |

Example

JSON Mode

```json
{
 "name":"John"
}
```

Age might be missing.

Structured Output

```python
Employee(
 name="John",
 age=28,
 department="HR"
)
```

Schema validation ensures required fields are present.

---

# 10. Enterprise Example

Resume Parser

Input

```
John Doe

Python Developer

5 Years
```

Output

```json
{
  "name":"John Doe",
  "experience":5,
  "skills":[
    "Python"
  ]
}
```

Easy to insert into a database.

---

# Part 2 : Streaming

# 11. What is Streaming?

Normally

```
User waits...

↓

Entire response arrives
```

Streaming sends tokens as they are generated.

---

# Interview Definition ⭐⭐⭐⭐⭐

> **Streaming is the incremental delivery of an LLM's output as it is generated, improving responsiveness and user experience.**

---

# 12. Without Streaming

```text
User

↓

LLM

↓

10 Seconds

↓

Entire Answer
```

---

# 13. With Streaming

```text
User

↓

LLM

↓

Token

↓

Token

↓

Token

↓

Complete Answer
```

The user starts reading immediately.

---

# 14. Streaming Example

```python
for chunk in llm.stream(
    "Explain LangChain."
):
    print(chunk.content, end="")
```

Output

```
LangChain
is
an
open-source
framework...
```

---

# 15. Streaming in LCEL

LCEL supports streaming naturally.

```python
chain = prompt | llm

for chunk in chain.stream(
    "Explain RAG."
):
    print(chunk.content, end="")
```

---

# 16. Streaming Workflow

```text
Prompt

↓

LLM

↓

Token 1

↓

Token 2

↓

Token 3

↓

Finished
```

---

# 17. Streaming Events

During streaming

```
Start

↓

Receive Token

↓

Receive Token

↓

Receive Token

↓

End
```

Callbacks can observe these events for monitoring or UI updates.

---

# 18. Structured Output + Streaming

Can they work together?

Usually yes, but with care.

Example

```text
Streaming

↓

Partial JSON

↓

Final JSON

↓

Validation
```

Many production systems wait until the full response is received before validating the structured output.

---

# 19. Production Use Cases

### Structured Output

- Resume Parsing
- Invoice Processing
- Information Extraction
- Order Processing
- CRM Automation
- API Responses

---

### Streaming

- Chatbots
- AI Assistants
- Code Generation
- Long Reports
- Content Writing

---

# 20. Best Practices

### Structured Output

✅ Define clear schemas.

✅ Validate required fields.

✅ Handle validation failures gracefully.

---

### Streaming

✅ Stream long responses.

✅ Show typing indicators in the UI.

✅ Handle interrupted connections.

---

# 21. Common Mistakes

❌ Parsing free text with regex when structured output is available.

❌ Returning malformed JSON.

❌ Streaming tiny responses unnecessarily.

❌ Assuming partial streamed JSON is complete.

---

# 22. Interview Questions

## Q1. What is Structured Output?

**Answer:**

Structured Output constrains the LLM to return data in a predefined schema, making it reliable for downstream applications.

---

## Q2. Why use Structured Output instead of parsing text?

**Answer:**

Parsing free text is brittle and error-prone. Structured Output provides predictable, validated data that applications can consume directly.

---

## Q3. Difference between JSON Mode and Structured Output?

**Answer:**

JSON Mode guarantees JSON formatting, while Structured Output enforces both the format and the schema, including expected fields and data types.

---

## Q4. What is Streaming?

**Answer:**

Streaming delivers the model's response incrementally as tokens are generated, improving responsiveness and perceived performance.

---

## Q5. When should you use Streaming?

**Answer:**

Streaming is ideal for long responses, chat applications, and interactive assistants where users benefit from seeing output immediately.

---

## Q6. Can Structured Output and Streaming be combined?

**Answer:**

Yes, but validation generally occurs after the complete structured response has been received. Applications should avoid acting on incomplete streamed JSON.

---

# 23. Quick Revision

| Concept | Purpose |
|----------|----------|
| Structured Output | Reliable typed responses |
| JSON Mode | Valid JSON |
| Pydantic | Schema definition |
| Streaming | Token-by-token output |
| stream() | Incremental generation |

---

# Interview Cheat Sheet

```text
Structured Output

Prompt

↓

LLM

↓

Schema Validation

↓

Typed Object

======================

Streaming

Prompt

↓

LLM

↓

Token

↓

Token

↓

Token

↓

Complete Response
```

---

# 24. Real Interview Scenario

**Question:**

> You are building an invoice extraction system. The LLM sometimes returns different field names such as `amount`, `totalAmount`, or `invoice_total`, causing downstream failures. How would you solve this?

**Answer:**

I would use Structured Output with a predefined schema (for example, a Pydantic model) that explicitly defines fields such as `invoice_number`, `vendor_name`, `total_amount`, and `invoice_date`. The LLM would be constrained to return data matching this schema, reducing parsing errors and making the output reliable for APIs and databases. If validation fails, I would retry or return a controlled error.

---

# 30-Second Interview Answer

> **Structured Output enables LLMs to return validated JSON or typed objects instead of unpredictable free text, making AI applications more reliable and easier to integrate with downstream systems. Streaming delivers responses incrementally as tokens are generated, reducing perceived latency and improving user experience. Together, they are essential features for building robust, production-grade AI applications.**

---

# Key Takeaway

> **Structured Output makes AI responses machine-friendly, while Streaming makes them user-friendly. Modern production AI systems often use both together to achieve reliability, responsiveness, and seamless integration with other services.**